<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Simulation Modeling Using Simpy (Part 1)

Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)


<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

# Recap: Mensa Simulation

In the last lecture, you saw a **complete simulation study** of the university Mensa. The simulation used several SimPy concepts:

| What the simulation did | SimPy concept used |
|---|---|
| Modeled cash registers with limited capacity | `simpy.Resource` |
| Modeled students arriving, waiting, being served | Processes (`yield`, `env.timeout`) |
| Generated students over time | Generator function |
| Collected waiting times and queue lengths | Results dictionary |
| Compared 3 vs. 4 registers | Multiple replications + confidence intervals |

You saw the **end product** — a working simulation that answered a real question. But we didn't explain **how SimPy actually works under the hood**. That's the topic of the following lectures, starting today.

# Agenda

* **Discrete-event simulation:** What is it and why does it matter for business decisions?
* **SimPy basics:** How do processes and the event loop work? (`yield`, `env.timeout`)
* **Process interactions:** How can one process wait for another?
* **Shared resources:** How do multiple processes compete for limited capacity?

All of these appeared in the Mensa simulation — today we'll build up the understanding **step by step**, starting from the simplest possible examples.

Credits: The following content is adapted from the official [Simpy documentation](https://simpy.readthedocs.io/en/latest/simpy_intro/index.html)

# What is Discrete-Event Simulation?

The Mensa model is an example of a **discrete-event simulation (DES)**. What does that mean?

In the Mensa, the system doesn't change continuously — it changes only when **specific things happen**:
- A student **arrives** → queue length increases
- A register becomes **free** → the next student starts being served
- A service is **finished** → the student leaves

Each of these is an **event**. Between events, nothing changes — the system just sits in its current state. The simulation clock **jumps** from one event to the next, skipping over the idle time in between.

There are several types of simulation. The following table gives an overview:

| Type | Key idea | Time | Typical applications | Tools |
|---|---|---|---|---|
| **Discrete-Event Simulation** | State changes only at specific events | Jumps to next event | Queuing systems, logistics, manufacturing, service systems | SimPy |
| **Continuous Simulation** | State changes at every point in time | Fixed small time steps | Physics, weather, fluid dynamics | Differential equation solvers |
| **Monte Carlo Simulation** | Repeated random sampling to estimate outcomes | No time dimension | Risk analysis, finance, numerical integration | NumPy, R |
| **Agent-Based Simulation** | Autonomous agents follow individual rules; focus on emergent behavior | Discrete or continuous | Epidemics, traffic, markets, social systems | Mesa, NetLogo |
| **System Dynamics** | Aggregate stocks, flows and feedback loops | Continuous | Policy analysis, population dynamics, business strategy | Vensim, Stella |

In this course, we focus exclusively on **discrete-event simulation**. SimPy is a Python library designed specifically for this purpose.

**Why is DES the right approach for business decisions?**

Most operational business systems — service counters, supply chains, manufacturing lines, call centers, hospitals — share a common structure: **entities** (customers, orders, patients, ...) arrive, compete for **limited resources**, get processed, and leave. What matters for decision-making are the **outcomes** of these interactions: How long did customers wait? How many were served? How utilized were the resources?

These outcomes only change when something happens (an arrival, a service completion, a breakdown). What happens *between* events is irrelevant — nobody cares what the queue "looks like" at 10:03:27 if nothing changed since 10:03:15. DES exploits this: by jumping from event to event, it can simulate hours, days, or weeks of operation in seconds, while still capturing everything that matters for the decision.

# Today: Model Translation

<img src="images/simulation_study_steps_translation.png" style="width:100%; float:left;" />

A simulation study follows a structured process from problem formulation to implementation. This lecture focuses on the **Model Translation** step — turning a conceptual model into a running computer simulation using SimPy.

- **Input:** A conceptual model (e.g., "students arrive, queue, get served")
- **Output:** A running computer simulation using SimPy

## Example: Conceptual Model of the Mensa

Before we can write code, we need a clear picture of how the system works. The conceptual model describes the flow of entities through the system — here, students arriving with their meal, waiting for a free cashier, paying, and leaving.

<img src="images/conceptual_model_mensa.png" style="width:80%;" />

## From Conceptual Model to Code

Looking at the flowchart, we can already identify what we need to model:

- **Entities** that move through the system (students)
- **Actions that take time** (paying at the cashier)
- **Limited resources** that cause waiting (cashiers)
- **Arrivals** that generate new entities over time

SimPy provides three core building blocks that map directly to these elements. Let's give them proper names.

## Why SimPy?

**SimPy** is a Python library for discrete-event simulation. Its main advantages:

- **Pure Python** — no special graphical user interface (GUI) or compiler needed, just `pip install simpy`
- **Lightweight** — simulations are written as standard Python functions using `yield`
- **Flexible** — full access to Python's ecosystem (NumPy, pandas, matplotlib) for data analysis and visualization
- **Open source** and well-documented

Other common simulation tools:

| Tool | Type | Language | Notes |
|---|---|---|---|
| **SimPy** | DES library | Python | Code-based, lightweight, free |
| **AnyLogic** | Multi-method | Java/GUI | Commercial, powerful GUI, agent-based + DES |
| **Arena** | DES | GUI | Commercial, widely used in industry |
| **Simio** | DES + agent-based | GUI | Commercial, 3D visualization |
| **MATLAB/Simulink** | Continuous + DES | MATLAB | Commercial, strong in engineering |
| **Salabim** | DES library | Python | Similar to SimPy, built-in animation |

We use SimPy because it integrates naturally with the Python data science stack and teaches you the **fundamentals** of DES without hiding them behind a GUI.

In [1]:
import simpy

## SimPy: The Core Building Blocks

| Building block | What it does | Mensa example | SimPy code |
|---|---|---|---|
| **Process** | A sequence of actions that unfolds over time | The student's flow: arrive → join queue → pay → leave | `def run(self): ...` with `yield` statements |
| **Event** | Something that happens at a specific point in time | A student arrives, a cashier becomes free, a payment is completed | `env.timeout(duration)`, `resource.request()` |
| **Environment** | Manages the simulation clock and schedules all events | Keeps track of time and decides which event happens next | `env = simpy.Environment()`, `env.run()` |

To understand how these work together, we'll start with the **simplest possible process** — much simpler than the Mensa.

## Our First Process

Let's start with the simplest possible SimPy process: a car that alternates between parking and driving.

<img src="images/flowchart_park_drive.png" style="width:70%;" />

We'll show the code first, run it, and then explain how it works.

In [3]:
parking_duration = 5
trip_duration = 2

def park_and_drive(env): #definition of the process (like a function, but without "return" statement)
    while True:
        print('Car starts parking at %d' % env.now)
        yield env.timeout(parking_duration) #pause the process until timeout (=parking) is over, then continue

        print('Car starts driving at %d' % env.now)
        yield env.timeout(trip_duration) #pause the process until timeout (=driving) is over, then continue

To run this process, we need to: (1) create a SimPy **environment**, (2) **register** the process, and (3) **start** the simulation:

In [4]:
env = simpy.Environment() #create environment
env.process(park_and_drive(env)) #add the "park_and_drive" process to the environment 

<Process(park_and_drive) object at 0x109effb00>

In [5]:
env.run(until=30)

Car starts parking at 0
Car starts driving at 5
Car starts parking at 7
Car starts driving at 12
Car starts parking at 14
Car starts driving at 19
Car starts parking at 21
Car starts driving at 26
Car starts parking at 28


### What just happened? The SimPy Event Loop

The output above was produced by SimPy's **event loop** (also called the **scheduler**). Here's what it did, step by step:

| Step | Simulation clock | Event triggered | Action | Next event scheduled |
|---|---|---|---|---|
| 1 | t = 0 | Process starts | Print "starts parking at 0" | Timeout at t = 5 (parking ends) |
| 2 | t = 5 | Timeout fires | Print "starts driving at 5" | Timeout at t = 7 (driving ends) |
| 3 | t = 7 | Timeout fires | Print "starts parking at 7" | Timeout at t = 12 |
| 4 | t = 12 | Timeout fires | Print "starts driving at 12" | Timeout at t = 14 |
| ... | ... | ... | ... | ... |
| | t = 30 | `until` reached | Simulation stops | — |

Notice:
- The clock **jumps** directly from 0 → 5 → 7 → 12 → ... There is no t = 1, t = 2, etc. This is what makes it a **discrete-event** simulation.
- At each step, the scheduler (1) advances the clock to the next event, (2) resumes the process, (3) the process runs until it hits the next `yield`, which schedules a new event, and (4) the process is suspended again.
- The `yield` statement is the boundary: everything before it executes immediately, then the process **pauses** until the yielded event occurs.

### Understanding `yield` in SimPy

The keyword `yield` is what makes SimPy processes work. Here's all you need to know:

- **`yield env.timeout(duration)`** — pause this process for `duration` time units, then resume
- **`yield env.process(...)`** — pause this process until another process finishes (we'll see this soon)
- **`yield resource.request()`** — pause this process until a resource becomes available (we'll see this soon)

In every case, the pattern is the same: `yield` **hands control back to the scheduler** and tells it *when* to resume the process.

**Program flow:**

<img src="https://raw.githubusercontent.com/GuntherGust/S4DM_data/main/program_flow_yield.png" style="width:70%; float:center;" />

### Excursus: Python Generators

Under the hood, SimPy processes are Python **generators**. A generator is a special kind of function that uses `yield` instead of `return`:

- A normal function runs to completion and returns a value.
- A generator function **pauses** at each `yield` statement, **returns a value** to the caller, and **resumes** where it left off when called again.

In SimPy, the value returned by `yield` is always an **event** — it tells the scheduler *what the process is waiting for* (e.g., a timeout, a resource request). The scheduler receives this event, schedules it, and resumes the process once the event is triggered.

**Syntactically**, the only difference is `yield` instead of `return`. **Conceptually**, think of it like a bookmark in a book: `yield` marks the page and hands a note to the scheduler ("wake me up in 5 minutes"), you close the book (the scheduler runs other processes), and when you reopen it, you continue exactly where you left off.

### Mentimeter

## Process Interaction

So far, our car process used only `yield env.timeout(...)` — it waited for time to pass. But what if a process needs to **wait for another process to finish**?

A common example: an electric vehicle must **charge** before it can drive again. Charging is a separate activity with its own duration. The driving process needs to wait until charging is complete.

We model this by giving the `Car` class two process methods: `park_and_drive()` and `charge()`. The key line is `yield self.env.process(self.charge(...))` — it starts the charging process and **waits for it to finish** before continuing.

In [6]:
class Car:
    def __init__(self, env): #"constructor" method that is invoked when "Car()" in called
        self.env = env
        # Start the park_and_drive process everytime a car instance is created.
        self.action = env.process(self.park_and_drive())

    def park_and_drive(self):
        while True:
            print('Start parking and charging at %d' % self.env.now)
            charge_duration = 5
            # We yield the process that process() returns
            # to wait for it to finish
            yield self.env.process(self.charge(charge_duration))

            # The charge process has finished and
            # we can start driving again.
            print('Start driving at %d' % self.env.now)
            trip_duration = 2
            yield self.env.timeout(trip_duration)

    def charge(self, duration):
        yield self.env.timeout(duration)

In [8]:
env2 = simpy.Environment()
car = Car(env2)
car2 = Car(env2)
env2.run(until=33)

Start parking and charging at 0
Start parking and charging at 0
Start driving at 5
Start driving at 5
Start parking and charging at 7
Start parking and charging at 7
Start driving at 12
Start driving at 12
Start parking and charging at 14
Start parking and charging at 14
Start driving at 19
Start driving at 19
Start parking and charging at 21
Start parking and charging at 21
Start driving at 26
Start driving at 26
Start parking and charging at 28
Start parking and charging at 28


### `yield env.timeout(...)` vs. `yield env.process(...)`

We now have two ways to pause a process:

| Statement | What it does | Resumption time |
|---|---|---|
| `yield env.timeout(duration)` | Pause for a fixed amount of time | **Known** in advance (current time + duration) |
| `yield env.process(other_process)` | Pause until another process finishes | **May be unknown** — depends on what the other process does |

This distinction matters: with `yield env.timeout(5)`, the scheduler knows exactly when to resume (t + 5). But with `yield env.process(self.charge(...))`, the resumption time depends on the entire execution of the charging process — which could itself involve waiting for resources, random durations, or other processes. The calling process simply says: "wake me up when it's done, however long that takes."

## Shared Resources

We've seen two types of `yield` so far: waiting for **time** and waiting for **another process**. But in the Mensa simulation, students also waited for a **cash register to become free**. This is the third type: waiting for a **shared resource** with limited capacity.

Let's revisit the EV example. Now multiple cars drive to a charging station that has only **2 charging spots**. If both spots are occupied, a car must **wait in a queue**.

In [9]:
class Car:
    def __init__(self, env, name):
        self.env = env
        self.name = name
        
    def drive(self, station, driving_time, charge_duration):
        
        # Simulate driving to the charging station
        yield self.env.timeout(driving_time)

          # Request one of its charging spots
        print('%s arriving at the charging station at %d' % (self.name, self.env.now))
        
        with station.request() as req:
                yield req #wait until a spot is available

                # Charge the battery
                print('%s starting to charge at %s' % (self.name, self.env.now))

                 yield self.env.timeout(charge_duration)
                print('%s leaving the charging station at %s' % (self.name, self.env.now))

In [25]:
env3 = simpy.Environment()
station = simpy.Resource(env3, capacity=8)

In [26]:
for i in range(1, 11):
    car = Car(env=env3, name='Car %d' % i)
    env3.process(car.drive(station = station, driving_time = i*2, charge_duration =  20))

In [27]:
env3.run()

Car 1 arriving at the charging station at 2
Car 1 starting to charge at 2
Car 2 arriving at the charging station at 4
Car 2 starting to charge at 4
Car 3 arriving at the charging station at 6
Car 3 starting to charge at 6
Car 4 arriving at the charging station at 8
Car 4 starting to charge at 8
Car 5 arriving at the charging station at 10
Car 5 starting to charge at 10
Car 6 arriving at the charging station at 12
Car 6 starting to charge at 12
Car 7 arriving at the charging station at 14
Car 7 starting to charge at 14
Car 8 arriving at the charging station at 16
Car 8 starting to charge at 16
Car 9 arriving at the charging station at 18
Car 10 arriving at the charging station at 20
Car 1 leaving the charging station at 22
Car 9 starting to charge at 22
Car 2 leaving the charging station at 24
Car 10 starting to charge at 24
Car 3 leaving the charging station at 26
Car 4 leaving the charging station at 28
Car 5 leaving the charging station at 30
Car 6 leaving the charging station at 32


### How resources work

The key pattern in the code is:

```python
with station.request() as req:
    yield req                          # wait until a spot is free
    yield self.env.timeout(duration)   # use the spot
                                       # spot is released automatically
```

- **`station.request()`** creates a request event. If a spot is free, the process continues immediately. If not, it **waits in a queue** (FIFO — first in, first out).
- The **`with` statement** ensures the resource is automatically released when the block ends.
- When a spot is released, the **next waiting process** in the queue is resumed.

### Summary: Three types of `yield`

| Statement | Waits for | Resumption time |
|---|---|---|
| `yield env.timeout(duration)` | Time to pass | **Known** (current time + duration) |
| `yield env.process(...)` | Another process to finish | **Unknown** — depends on sub-process |
| `yield resource.request()` | A resource to become available | **Unknown** — depends on other users |

### Mentimeter

## Summary

| Concept | What you learned |
|---|---|
| **Discrete-event simulation** | State changes only at events; the clock jumps between them |
| **`yield env.timeout(duration)`** | Pause a process for a known amount of time |
| **`yield env.process(...)`** | Pause until another process finishes (unknown duration) |
| **`yield resource.request()`** | Pause until a shared resource is available (unknown duration) |
| **Event loop** | The scheduler advances the clock, resumes the next process, and suspends it again at the next `yield` |

## Outlook: Next Lecture

Now that you understand the mechanics of SimPy, the next lecture will focus on **how to structure simulation models**:

- **Modeling conventions:** A systematic way to organize entities, resources, and generators into classes
- **Object-oriented programming:** The basics you need to write your own simulation models
- **A complete example:** The carwash simulation — building a structured model from scratch

## Further materials for practice

* See the [Simpy documentation]() for more materials
* __Object-oriented prorgamming:__
    * For a very very short introduction to the syntax we use, see e.g. [this video](https://www.youtube.com/watch?v=JH4q65dZPvY&ab_channel=Indently)
    * For more thourough practice, we provide a link to a __DataCamp course__ on Wuecampus.

<img src="images/d3.png" style="width:50%; float:center;" />
